# <center>Remaining Useful Life (RUL) Prediction for Aircraft Engine Predictive Maintenance</center>

In [19]:
# ============================================================
# Aircraft Engine RUL Prediction Model Training
# Models Used: SVR, Random Forest, XGBoost
# Dataset: NASA C-MAPSS FD001
# ============================================================


## Introduction

This project focuses on predicting the Remaining Useful Life (RUL) of aircraft engines using machine learning. RUL represents the estimated number of operating cycles an engine can continue to function before failure or maintenance is required.

The dataset used is the NASA C-MAPSS FD001 dataset, which contains time-series sensor readings from multiple aircraft engines operating until failure. Each engine has operational settings and sensor measurements recorded across different cycles. The goal is to train machine learning models to learn degradation patterns from these sensor values and predict the engine’s remaining useful life.

The project also includes a Streamlit dashboard, where users can enter a small number of important sensor values and receive a prediction along with a health status, key drivers, interpretation, and maintenance recommendation.

## Pipeline

    NASA C-MAPSS Dataset
        ↓
    Data Loading and Conversion
        ↓
    RUL Calculation
        ↓
    Feature Selection
        ↓
    Engine-Level Train-Validation Split
        ↓
    Feature Scaling
        ↓
    Model Training
    SVR | Random Forest | XGBoost
        ↓
    Model Evaluation
    MAE | MSE | RMSE | R² Score
        ↓
    Best Model Selection
        ↓
    Save Model Files
        ↓
    Streamlit Dashboard
        ↓
    Prediction Summary, Key Drivers, Interpretation, Recommendation

## Import Libraries

In [2]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor

## Convert NASA C-MAPSS TXT files to CSV

The project uses the NASA C-MAPSS FD001 dataset. The raw .txt files were converted into .csv format for easier handling in Python.

In [3]:
base_path = r"C:\Users\ashik\Downloads\archive (3)\CMaps"

train_df = pd.read_csv(os.path.join(base_path, "train_FD001.csv"))
test_df = pd.read_csv(os.path.join(base_path, "test_FD001.csv"))
rul_df = pd.read_csv(os.path.join(base_path, "RUL_FD001.csv"))

print("CSV files loaded successfully")
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("RUL shape:", rul_df.shape)

train_df.head()

CSV files loaded successfully
Train shape: (20631, 26)
Test shape: (13096, 26)
RUL shape: (100, 1)


,engine_id,cycle,op1,op2,op3,sensor1,sensor2,sensor3,sensor4,sensor5,...,sensor12,sensor13,sensor14,sensor15,sensor16,sensor17,sensor18,sensor19,sensor20,sensor21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


## Create RUL Column for Training Data

The original training data does not directly provide the Remaining Useful Life for each row. Therefore, RUL is calculated manually.

For each engine, the maximum cycle is identified. Then RUL is calculated as:

<b>RUL = Maximum cycle of engine - Current cycle</b>

Example:

If an engine fails at cycle 192
and the current cycle is 1,
then RUL = 192 - 1 = 191

In [4]:
# Find max cycle for each engine
max_cycle = train_df.groupby("engine_id")["cycle"].max().reset_index()
max_cycle.columns = ["engine_id", "max_cycle"]

# Merge max cycle with train data
train_df = train_df.merge(max_cycle, on="engine_id", how="left")

# Remaining Useful Life = max cycle - current cycle
train_df["RUL"] = train_df["max_cycle"] - train_df["cycle"]

# Cap RUL at 130 for stable prediction
train_df["RUL_capped"] = train_df["RUL"].clip(upper=130)

# Remove helper column
train_df.drop(columns=["max_cycle"], inplace=True)

print("RUL created successfully")
train_df[["engine_id", "cycle", "RUL", "RUL_capped"]].head()

RUL created successfully


,engine_id,cycle,RUL,RUL_capped
0,1,1,191,130
1,1,2,190,130
2,1,3,189,130
3,1,4,188,130
4,1,5,187,130


## Feature Selection

Columns such as engine_id, cycle, RUL, and RUL_capped are not used as model inputs.

In [5]:
drop_columns = ["engine_id", "cycle", "RUL", "RUL_capped"]

features = [col for col in train_df.columns if col not in drop_columns]

X = train_df[features]
y = train_df["RUL_capped"]

print("Features used:")
print(features)

print("\nNumber of features:", len(features))
print("X shape:", X.shape)
print("y shape:", y.shape)

Features used:
['op1', 'op2', 'op3', 'sensor1', 'sensor2', 'sensor3', 'sensor4', 'sensor5', 'sensor6', 'sensor7', 'sensor8', 'sensor9', 'sensor10', 'sensor11', 'sensor12', 'sensor13', 'sensor14', 'sensor15', 'sensor16', 'sensor17', 'sensor18', 'sensor19', 'sensor20', 'sensor21']

Number of features: 24
X shape: (20631, 24)
y shape: (20631,)


## Engine-Level Train-Validation Split

Instead of randomly splitting rows, the dataset is split by engine_id.

This is important because the same engine should not appear in both training and validation data. If the same engine appears in both, the model may indirectly learn that engine’s pattern and give overly optimistic results.

In [6]:
engine_ids = train_df["engine_id"].unique()

train_engines, val_engines = train_test_split(
    engine_ids,
    test_size=0.2,
    random_state=42
)

train_data = train_df[train_df["engine_id"].isin(train_engines)]
val_data = train_df[train_df["engine_id"].isin(val_engines)]

X_train = train_data[features]
y_train = train_data["RUL_capped"]

X_val = val_data[features]
y_val = val_data["RUL_capped"]

print("Engine-level split completed")
print("Training engines:", len(train_engines))
print("Validation engines:", len(val_engines))
print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)

Engine-level split completed
Training engines: 80
Validation engines: 20
X_train shape: (16561, 24)
X_val shape: (4070, 24)


## Feature Scaling

Feature scaling is applied using StandardScaler.

This transforms the input values so that each feature has a similar scale. Scaling is especially important for SVR, because SVR is sensitive to the magnitude of input values.

The scaler is fitted only on the training data and then applied to the validation data.

In [7]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print("Feature scaling completed")

Feature scaling completed


# Model Training

<b>Three machine learning models are trained and compared.</b>

<h3>SVR</h3>
<h3>RandomForest</h3>
<h3>XGBoost</h3>

## SVR

Support Vector Regression is used with an RBF kernel. It is useful for capturing non-linear relationships in the data.

In [8]:
svr_model = SVR(
    kernel="rbf",
    C=100,
    gamma="scale",
    epsilon=0.1
)

print("Training SVR model...")

svr_model.fit(X_train_scaled, y_train)

svr_pred = svr_model.predict(X_val_scaled)

svr_mae = mean_absolute_error(y_val, svr_pred)
svr_mse = mean_squared_error(y_val, svr_pred)
svr_rmse = np.sqrt(svr_mse)
svr_r2 = r2_score(y_val, svr_pred)

print("SVR Model Results")
print("MAE:", svr_mae)
print("MSE:", svr_mse)
print("RMSE:", svr_rmse)
print("R2 Score:", svr_r2)

Training SVR model...
SVR Model Results
MAE: 12.455468945417108
MSE: 352.90650843002487
RMSE: 18.78580603620789
R2 Score: 0.8132279177579929


## Random Forest Regressor

Random Forest is an ensemble learning method that builds multiple decision trees and averages their predictions.

In [9]:
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

print("Training Random Forest model...")

rf_model.fit(X_train_scaled, y_train)

rf_pred = rf_model.predict(X_val_scaled)

rf_mae = mean_absolute_error(y_val, rf_pred)
rf_mse = mean_squared_error(y_val, rf_pred)
rf_rmse = np.sqrt(rf_mse)
rf_r2 = r2_score(y_val, rf_pred)

print("Random Forest Model Results")
print("MAE:", rf_mae)
print("MSE:", rf_mse)
print("RMSE:", rf_rmse)
print("R2 Score:", rf_r2)

Training Random Forest model...
Random Forest Model Results
MAE: 13.28795309613146
MSE: 327.87604116693376
RMSE: 18.107347712101124
R2 Score: 0.8264750310260807


## XGBoost Regressor

XGBoost is a gradient boosting algorithm that builds trees sequentially, where each new tree tries to correct the errors of the previous trees.

In [10]:
xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

print("Training XGBoost model...")

xgb_model.fit(X_train_scaled, y_train)

xgb_pred = xgb_model.predict(X_val_scaled)

xgb_mae = mean_absolute_error(y_val, xgb_pred)
xgb_mse = mean_squared_error(y_val, xgb_pred)
xgb_rmse = np.sqrt(xgb_mse)
xgb_r2 = r2_score(y_val, xgb_pred)

print("XGBoost Model Results")
print("MAE:", xgb_mae)
print("MSE:", xgb_mse)
print("RMSE:", xgb_rmse)
print("R2 Score:", xgb_r2)

Training XGBoost model...
XGBoost Model Results
MAE: 13.072125434875488
MSE: 321.6097717285156
RMSE: 17.93348186294328
R2 Score: 0.8297914266586304


## Evaluation Metrics

The models were evaluated using four metrics.

<h4>MAE</h4>

Mean Absolute Error measures the average absolute difference between actual and predicted RUL.

Lower MAE means better performance.

<h4>MSE</h4>

Mean Squared Error squares the prediction errors before averaging them. It penalises larger errors more strongly.

Lower MSE means better performance.

<h4>RMSE</h4>

Root Mean Squared Error is the square root of MSE. It is easier to interpret because it is in the same unit as the target variable.

Lower RMSE means better performance.

<h4>R² Score</h4>

R² Score shows how much variance in the target variable is explained by the model.

Higher R² means better performance.

In [11]:
results = pd.DataFrame({
    "Model": ["SVR", "Random Forest", "XGBoost"],
    "MAE": [svr_mae, rf_mae, xgb_mae],
    "MSE": [svr_mse, rf_mse, xgb_mse],
    "RMSE": [svr_rmse, rf_rmse, xgb_rmse],
    "R2 Score": [svr_r2, rf_r2, xgb_r2]
})

results = results.sort_values(by="RMSE", ascending=True)

print("Model Comparison:")
results

Model Comparison:


,Model,MAE,MSE,RMSE,R2 Score
2,XGBoost,13.072125,321.609772,17.933482,0.829791
1,Random Forest,13.287953,327.876041,18.107348,0.826475
0,SVR,12.455469,352.906508,18.785806,0.813228


## Best Model Selected

Based on the evaluation table, XGBoost was selected as the best model.

Although SVR achieved the lowest MAE, XGBoost achieved the best overall performance because it had:

In [12]:
best_model_name = results.iloc[0]["Model"]

if best_model_name == "SVR":
    best_model = svr_model
elif best_model_name == "Random Forest":
    best_model = rf_model
else:
    best_model = xgb_model

print("Best Model Selected:", best_model_name)

Best Model Selected: XGBoost


### <b>This means XGBoost gave the most balanced and reliable performance on the validation data.</b>

## Creating Default Engine Profiles and Saving Model Files for Streamlit Deployment

This code creates sample engine profiles for three health conditions: Healthy, Warning, and Critical. It then saves the trained model, scaler, feature list, and default profiles as .pkl files. These saved files are later used by the Streamlit dashboard to make predictions without retraining the model every time.


In [13]:
import os
import joblib

def get_profile_by_rul_range(df, min_rul, max_rul):
    sample = df[
        (df["RUL_capped"] >= min_rul) &
        (df["RUL_capped"] <= max_rul)
    ]

    if sample.empty:
        return None

    sample = sample.iloc[0]

    profile = {}

    for feature in features:
        profile[feature] = float(sample[feature])

    return profile


default_profiles = {
    "Healthy sample": get_profile_by_rul_range(train_df, 90, 130),
    "Warning sample": get_profile_by_rul_range(train_df, 40, 80),
    "Critical sample": get_profile_by_rul_range(train_df, 0, 30),
}

default_profiles = {
    name: values
    for name, values in default_profiles.items()
    if values is not None
}

joblib.dump(best_model, os.path.join(base_path, "rul_model.pkl"))
joblib.dump(scaler, os.path.join(base_path, "scaler.pkl"))
joblib.dump(features, os.path.join(base_path, "features.pkl"))
joblib.dump(default_profiles, os.path.join(base_path, "default_profiles.pkl"))

print("Saved model files at:", base_path)
print("Profiles saved:", list(default_profiles.keys()))
print(os.listdir(base_path))

Saved model files at: C:\Users\ashik\Downloads\archive (3)\CMaps
Profiles saved: ['Healthy sample', 'Warning sample', 'Critical sample']
['Damage Propagation Modeling.pdf', 'default_profiles.pkl', 'features.pkl', 'model_report.csv', 'model_results.pkl', 'readme.txt', 'RUL_FD001.csv', 'RUL_FD001.txt', 'RUL_FD002.txt', 'RUL_FD003.txt', 'RUL_FD004.txt', 'rul_model.pkl', 'scaler.pkl', 'test_FD001.csv', 'test_FD001.txt', 'test_FD002.txt', 'test_FD003.txt', 'test_FD004.txt', 'train_FD001.csv', 'train_FD001.txt', 'train_FD002.txt', 'train_FD003.txt', 'train_FD004.txt', 'x.txt']


## Creating and Saving Default Engine Health Profiles

This code creates default input profiles for the Streamlit dashboard. These profiles represent three engine health conditions: Healthy, Warning, and Critical.

In [ ]:
# Select one sample from validation data
sample_input = X_val.iloc[[0]]

# Scale sample
sample_scaled = scaler.transform(sample_input)

# Predict RUL
sample_prediction = best_model.predict(sample_scaled)[0]

# Clip prediction between 0 and 130
sample_rul = int(np.clip(round(sample_prediction), 0, 130))

print("Sample Predicted RUL:", sample_rul)

if sample_rul > 80:
    status = "HEALTHY"
elif sample_rul > 30:
    status = "WARNING"
else:
    status = "CRITICAL"

print("Predicted Status:", status)

## Copying Saved Model Files to the Streamlit Project Folder

This code copies the required .pkl files from the model training folder to the Streamlit project folder.

In [ ]:
import os
import shutil

source_folder = r"C:\Users\ashik\Downloads\archive (3)\CMaps"
destination_folder = r"C:\Users\ashik\Project1"

files_to_copy = [
    "rul_model.pkl",
    "scaler.pkl",
    "features.pkl",
    "default_profiles.pkl"
]

for file in files_to_copy:
    source_path = os.path.join(source_folder, file)
    destination_path = os.path.join(destination_folder, file)

    if os.path.exists(source_path):
        shutil.copy(source_path, destination_path)
        print(f"Copied: {file}")
    else:
        print(f"Missing in source folder: {file}")

print("\nFiles now inside Project1:")
print(os.listdir(destination_folder))

## Conclusion

This project successfully developed a machine learning-based aircraft engine RUL prediction system using the NASA C-MAPSS FD001 dataset. The pipeline included data preprocessing, RUL calculation, feature selection, engine-level data splitting, scaling, model training, evaluation, model saving, and dashboard deployment.

Three models were compared: SVR, Random Forest, and XGBoost. Among them, XGBoost performed best, achieving the lowest RMSE and highest R² score. The final model was saved and integrated into a Streamlit dashboard.

The dashboard improves usability by allowing users to enter only key sensor values while automatically filling the remaining model features. It provides not only the predicted RUL, but also a health status, prediction summary, key drivers, interpretation, and maintenance recommendation.

Overall, the project demonstrates how machine learning can support predictive maintenance by identifying engine degradation early and helping users make better maintenance decisions.